In [11]:
!pip install -q langchain langchain-groq langchain-core \
pydantic pandas openpyxl

In [14]:
!pip install -U --upgrade-strategy eager langchain langchain-core langchain-groq

  Using cached uuid_utils-1.2.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)


In [16]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [19]:
!python -m pip uninstall -y langchain langchain-core langchain-groq

Found existing installation: langchain 1.5.2
Uninstalling langchain-1.5.2:
  Successfully uninstalled langchain-1.5.2
Found existing installation: langchain-core 1.8.5
Uninstalling langchain-core-1.8.5:
  Successfully uninstalled langchain-core-1.8.5
Found existing installation: langchain-groq 1.2.1
Uninstalling langchain-groq-1.2.1:
  Successfully uninstalled langchain-groq-1.2.1


In [20]:
!python -m pip install --no-cache-dir langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain]


In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key configured:", bool(os.environ["GROQ_API_KEY"]))

Groq API key configured: True


In [4]:
import os

print("API key exists:", bool(os.environ.get("GROQ_API_KEY")))

try:
    from langchain_groq import ChatGroq
    print("ChatGroq import: SUCCESS")

    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0
    )

    print("Groq initialization: SUCCESS")

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error:", str(e))

API key exists: True
Error type: ModuleNotFoundError
Error: No module named 'langchain_groq'


In [5]:
%pip install -q langchain-groq

In [18]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq initialized successfully!")

Groq initialized successfully!


In [19]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [20]:
from langchain.tools import tool

# Sample equipment information
equipment = {
    "camera": "working",
    "tripod": "working",
    "microphone": "working"
}

# Sample existing bookings
bookings = ["10:00", "10:15", "11:00"]


@tool
def check_equipment_status():
    """Check whether the camera, tripod and microphone are working."""

    if all(status == "working" for status in equipment.values()):
        return "All studio equipment is working."

    return f"Equipment status: {equipment}"


@tool
def check_kit_availability(time: str):
    """Check whether the studio kit is available at a given time."""

    if time in bookings:
        return f"The studio kit is already booked at {time}."

    return f"The studio kit is available at {time}."

In [21]:
print(check_equipment_status.invoke({}))

print(check_kit_availability.invoke({
    "time": "10:15"
}))

print(check_kit_availability.invoke({
    "time": "10:30"
}))

All studio equipment is working.
The studio kit is already booked at 10:15.
The studio kit is available at 10:30.


In [22]:
#Create the Pydantic model
from pydantic import BaseModel, Field
from typing import Optional


class BookingRecord(BaseModel):
    """Structured record for a studio kit booking enquiry."""

    customer_name: Optional[str] = Field(
        default=None,
        description="Name of the customer"
    )

    phone: Optional[str] = Field(
        default=None,
        description="Customer phone number"
    )

    booking_date: Optional[str] = Field(
        default=None,
        description="Requested booking date"
    )

    start_time: Optional[str] = Field(
        default=None,
        description="Requested booking start time"
    )

    duration_minutes: int = Field(
        default=15,
        description="Booking duration in minutes"
    )

    equipment: list[str] = Field(
        default_factory=lambda: [
            "camera", "tripod", "microphone"
        ],
        description="Equipment requested"
    )

    status: str = Field(
        default="pending",
        description="Initial enquiry status"
    )

In [11]:
#Connect Pydantic with Groq
structured_llm = llm.with_structured_output(
    BookingRecord,
    method="function_calling"
)

In [23]:
#Prepare 5 messy customer messages
messages = [
    "Hi, I am Rahul. Need the camera kit tomorrow at 10 AM.",

    "Bro, book the studio kit for me on Monday evening. "
    "My number is 9876543210.",

    "Need camera tripod mic for 15 mins at 2:30 PM "
    "on 28 November. Name: Kiran.",

    "Hello, can I get the kit tomorrow morning? "
    "I am Suresh and my contact is 9123456780.",

    "Please reserve the studio equipment for Friday at 4 PM. "
    "This is Priya, phone 9988776655."
]

In [24]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Groq model initialized successfully!")

Groq model initialized successfully!


In [16]:
response = llm.invoke("Introduce yourself in one sentence.")

print(response.content)

I’m ChatGPT, an AI language model created by OpenAI to help answer questions, generate ideas, and assist with a wide range of tasks.


In [17]:
structured_llm = llm.with_structured_output(
    BookingRecord,
    method="function_calling"
)

In [26]:
from pydantic import BaseModel, Field
from typing import Optional


class BookingRecord(BaseModel):

    customer_name: Optional[str] = Field(
        default=None,
        description="Customer name"
    )

    phone: Optional[str] = Field(
        default=None,
        description="Customer phone number"
    )

    booking_date: Optional[str] = Field(
        default=None,
        description="Requested booking date"
    )

    start_time: Optional[str] = Field(
        default=None,
        description="Requested booking start time"
    )

    duration_minutes: Optional[int] = Field(
        default=None,
        description="Booking duration in minutes"
    )

    equipment: Optional[list[str]] = Field(
        default=None,
        description="Requested equipment"
    )

    status: Optional[str] = Field(
        default=None,
        description="Booking enquiry status"
    )

In [28]:
structured_llm = llm.with_structured_output(
    BookingRecord,
    method="function_calling"
)

In [29]:
records = []

for i, message in enumerate(messages):

    try:
        result = structured_llm.invoke(
            "Extract booking information from this customer message. "
            "Do not invent missing information. "
            "Return null for unavailable fields. "
            "If the customer requests a camera kit, identify the "
            "equipment as camera, tripod and microphone only when "
            "the request clearly refers to the complete kit.\n\n"
            + message
        )

        records.append(result)

        print(f"Customer {i+1}")
        print(result.model_dump())
        print("-" * 60)

    except Exception as e:
        print(f"Customer {i+1} failed: {e}")

Customer 1
{'customer_name': 'Rahul', 'phone': None, 'booking_date': None, 'start_time': '10:00 AM', 'duration_minutes': None, 'equipment': ['camera', 'tripod', 'microphone'], 'status': None}
------------------------------------------------------------
Customer 2
{'customer_name': None, 'phone': '9876543210', 'booking_date': 'Monday', 'start_time': None, 'duration_minutes': None, 'equipment': None, 'status': None}
------------------------------------------------------------
Customer 3
{'customer_name': 'Kiran', 'phone': None, 'booking_date': '28 November', 'start_time': '2:30 PM', 'duration_minutes': 15, 'equipment': ['camera', 'tripod', 'microphone'], 'status': None}
------------------------------------------------------------
Customer 4
{'customer_name': 'Suresh', 'phone': '9123456780', 'booking_date': '2026-11-28', 'start_time': None, 'duration_minutes': None, 'equipment': None, 'status': None}
------------------------------------------------------------
Customer 5
{'customer_name':

In [30]:
#Extract structured records
records = []

for message in messages:

    result = structured_llm.invoke(
        "Extract the booking information from this message. "
        "Do not guess missing details. Use null when information "
        "is unavailable. Preserve relative dates if they cannot "
        "be resolved.\n\n"
        + message
    )

    records.append(result)

    print(result.model_dump())
    print("-" * 60)

{'customer_name': 'Rahul', 'phone': None, 'booking_date': 'tomorrow', 'start_time': '10 AM', 'duration_minutes': None, 'equipment': ['camera kit'], 'status': None}
------------------------------------------------------------
{'customer_name': None, 'phone': '9876543210', 'booking_date': 'Monday', 'start_time': 'evening', 'duration_minutes': None, 'equipment': ['studio kit'], 'status': None}
------------------------------------------------------------
{'customer_name': 'Kiran', 'phone': None, 'booking_date': '28 November', 'start_time': '2:30 PM', 'duration_minutes': 15, 'equipment': ['camera', 'tripod', 'mic'], 'status': None}
------------------------------------------------------------
{'customer_name': 'Suresh', 'phone': '9123456780', 'booking_date': 'tomorrow morning', 'start_time': 'morning', 'duration_minutes': None, 'equipment': None, 'status': None}
------------------------------------------------------------
{'customer_name': 'Priya', 'phone': '9988776655', 'booking_date': 'Fri

In [31]:
#Create a sheet for your business and a log_booking tool
#Create the Excel sheet
import pandas as pd

columns = [
    "customer_name",
    "phone",
    "booking_date",
    "start_time",
    "duration_minutes",
    "equipment",
    "status"
]

df = pd.DataFrame(columns=columns)

df.to_excel(
    "studio_bookings.xlsx",
    index=False
)

print("Studio booking sheet created successfully!")

Studio booking sheet created successfully!


In [32]:
#create the logging tool
from langchain.tools import tool
from openpyxl import load_workbook
import os


@tool
def log_booking(
    customer_name: str,
    phone: str,
    booking_date: str,
    start_time: str,
    duration_minutes: int
):
    """Save a customer's studio kit booking enquiry into the Excel sheet."""

    file_name = "studio_bookings.xlsx"

    new_record = [
        customer_name,
        phone,
        booking_date,
        start_time,
        duration_minutes,
        "Camera, Tripod, Microphone",
        "Pending"
    ]

    workbook = load_workbook(file_name)
    sheet = workbook.active

    sheet.append(new_record)

    workbook.save(file_name)

    return f"Booking enquiry for {customer_name} saved successfully."

In [33]:
#Test the logging tool
result = log_booking.invoke({
    "customer_name": "Rahul",
    "phone": "9876543210",
    "booking_date": "2026-11-28",
    "start_time": "10:00 AM",
    "duration_minutes": 15
})

print(result)

Booking enquiry for Rahul saved successfully.


In [34]:
#View the saved records
df = pd.read_excel("studio_bookings.xlsx")

print(df)

  customer_name       phone booking_date start_time  duration_minutes  \
0         Rahul  9876543210   2026-11-28   10:00 AM                15   

                    equipment   status  
0  Camera, Tripod, Microphone  Pending  


In [35]:
#Break it: Rename one tool to tool1 with docstring "does stuff"
#Rename the tool
@tool("tool1")
def check_kit_availability_bad(time: str):
    """does stuff"""

    if time in bookings:
        return f"The studio kit is already booked at {time}."

    return f"The studio kit is available at {time}."

In [36]:
#create an agent
from langchain.agents import create_agent

tools = [
    check_equipment_status,
    check_kit_availability_bad,
    log_booking
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a studio kit booking assistant. "
        "Use the appropriate tools to answer booking questions."
    )
)

In [37]:
#Ask the agent a question
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Is the studio kit available at 10:15?"
        }
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Is the studio kit available at 10:15?
================================== Ai Message ==================================

I’m happy to check that for you! Could you let me know which date you’d like to book the studio kit for 10:15 am? Once I have the date, I can confirm whether it’s available.


In [38]:
#change the tool bank
@tool("check_kit_availability")
def check_kit_availability(time: str):
    """Check whether the studio kit is available at a given time."""

    if time in bookings:
        return f"The studio kit is already booked at {time}."

    return f"The studio kit is available at {time}."

In [39]:
#Update the agent
tools = [
    check_equipment_status,
    check_kit_availability,
    log_booking
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a studio kit booking assistant. "
        "Use the appropriate tools to answer booking questions."
    )
)

In [40]:
#Read the trace: Count tool calls and model calls using pretty_print()
#Invoke the agent
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Check whether all equipment is working "
                "and whether the kit is available at 10:30."
            )
        }
    ]
})

In [41]:
#Display the trace
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Check whether all equipment is working and whether the kit is available at 10:30.
================================== Ai Message ==================================
Tool Calls:
  check_equipment_status (fc_63fcaf06-9635-4999-8a15-93e889182af3)
 Call ID: fc_63fcaf06-9635-4999-8a15-93e889182af3
  Args:
================================= Tool Message =================================
Name: check_equipment_status

All studio equipment is working.
================================== Ai Message ==================================
Tool Calls:
  check_kit_availability (fc_f46727a7-7c95-43fb-bf85-3baa1de0b9ac)
 Call ID: fc_f46727a7-7c95-43fb-bf85-3baa1de0b9ac
  Args:
    time: 10:30
================================= Tool Message =================================
Name: check_kit_availability

The studio kit is available at 10:30.
================================== Ai Message ==================================

✅ All equ

In [42]:
#Count model calls and tool calls
from langchain_core.messages import AIMessage, ToolMessage

model_calls = 0
tool_calls = 0

for message in response["messages"]:

    if isinstance(message, AIMessage):
        model_calls += 1
        tool_calls += len(message.tool_calls)

print("Total model calls:", model_calls)
print("Total tool calls:", tool_calls)

Total model calls: 3
Total tool calls: 2
